<a href="https://colab.research.google.com/github/suphanatchanlek30/Super-AI-Engineer-Season-6-Mini-Hackathon-Week-4-4-Heart-Disease-Prediction/blob/main/Heart_Disease_Prediction_600367_%E0%B8%A8%E0%B8%B8%E0%B8%A0%E0%B8%93%E0%B8%B1%E0%B8%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1

In [ ]:
!nvidia-smi

Fri Apr  3 15:46:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Cell 2: Install packages

In [ ]:
!pip -q install kaggle catboost scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 360.9 kB/s eta 0:00:00


Cell 3: Set Kaggle credentials

In [ ]:
from google.colab import files
files.upload()  # เลือกไฟล์ kaggle.json จากเครื่อง


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"suphanatchanlek","key":"8a8540eac53e577378dc97b638e143da"}'}

Cell 4: Move kaggle.json to correct location

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

Cell 5

In [ ]:
COMPETITION = "super-ai-engineer-ss-6-heart-disease-prediction"
DATA_DIR = "/content/heart_data"

Cell 6

In [ ]:
!mkdir -p {DATA_DIR}
!kaggle competitions download -c {COMPETITION} -p {DATA_DIR}
!unzip -o {DATA_DIR}/*.zip -d {DATA_DIR}
!ls -lh {DATA_DIR}


100% 4.30M/4.30M [00:00<00:00, 99.3MB/s]

Archive:  /content/heart_data/super-ai-engineer-ss-6-heart-disease-prediction.zip
  inflating: /content/heart_data/sample_submission.csv  
  inflating: /content/heart_data/test.csv  
  inflating: /content/heart_data/train.csv  
total 41M
-rw-r--r-- 1 root root 1017K Mar 28 14:08 sample_submission.csv
-rw-r--r-- 1 root root  4.3M Mar 28 14:08 super-ai-engineer-ss-6-heart-disease-prediction.zip
-rw-r--r-- 1 root root  8.8M Mar 28 14:08 test.csv
-rw-r--r-- 1 root root   27M Mar 28 14:08 train.csv


Cell 7



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    average_precision_score,
    fbeta_score,
    precision_score,
    recall_score
)


Cell 8

In [ ]:
train_path = f"{DATA_DIR}/train.csv"
test_path = f"{DATA_DIR}/test.csv"
sample_path = f"{DATA_DIR}/sample_submission.csv"

train = pd.read_csv(train_path, encoding="utf-8-sig")
test = pd.read_csv(test_path, encoding="utf-8-sig")
sample_sub = pd.read_csv(sample_path, encoding="utf-8-sig")

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_sub.shape)

display(train.head())
display(test.head())
display(sample_sub.head())


train shape: (223084, 20)
test shape: (74361, 19)
sample_submission shape: (74361, 2)


,ID,History of HeartDisease or Attack,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,train_000001,No,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,train_000002,No,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,train_000003,No,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,train_000004,No,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
4,train_000005,NaN,Yes,Yes,Yes,34.56,Yes,No,No,Yes,No,Yes,Yes,Very Poor,Yes,Male,Some high school,"$15,000 to less than $20,000",98,Yes


,ID,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,test_000001,Yes,Yes,Yes,24.84,No,No,No,Yes,No,Yes,No,Good,No,Female,Some college or technical school,"$20,000 to less than $25,000",71,Yes
1,test_000002,Yes,No,Yes,29.08,Yes,No,No,No,No,Yes,No,Fair,No,Female,College graduate,"$50,000 to less than $75,000",61,No
2,test_000003,Yes,Yes,Yes,35.23,Yes,No,No,No,No,Yes,No,Fair,Yes,Female,Some college or technical school,"Less than $10,000",67,Yes
3,test_000004,No,No,Yes,24.78,Yes,No,No,No,No,Yes,No,Fair,No,Female,Some college or technical school,"$50,000 to less than $75,000",50,Yes
4,test_000005,No,No,Yes,27.57,Yes,No,No,No,No,Yes,No,Fair,No,Male,Some college or technical school,"$25,000 to less than $35,000",40,Yes


,ID,History of HeartDisease or Attack
0,test_000001,No
1,test_000002,No
2,test_000003,No
3,test_000004,NaN
4,test_000005,NaN


Cell 9

In [ ]:
target_col = "History of HeartDisease or Attack"
id_col = train.columns[0]

train = train[train[target_col].isin(["Yes", "No"])].copy()
train["target"] = (train[target_col] == "Yes").astype(int)

X = train.drop(columns=[target_col, "target", id_col]).copy()
X_test = test.drop(columns=[test.columns[0]]).copy()
y = train["target"].values

print("filtered train shape:", train.shape)
print("positive rate:", y.mean())


filtered train shape: (221390, 21)
positive rate: 0.08161163557522923


Cell 10

In [ ]:
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

print("categorical columns:", cat_cols)
print("numerical columns:", num_cols)


categorical columns: ['High Blood Pressure', 'Told High Cholesterol', 'Cholesterol Checked', 'Smoked 100+ Cigarettes', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Leisure Physical Activity', 'Heavy Alcohol Consumption', 'Health Care Coverage', 'Doctor Visit Cost Barrier', 'General Health', 'Difficulty Walking', 'Sex', 'Education Level', 'Income Level', 'Vegetable or Fruit Intake (1+ per Day)']
numerical columns: ['Body Mass Index', 'Age']


Cell 11

In [ ]:
for c in cat_cols:
    X[c] = X[c].fillna("MISSING").astype(str)
    X_test[c] = X_test[c].fillna("MISSING").astype(str)

for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")
    X_test[c] = pd.to_numeric(X_test[c], errors="coerce")

cat_features_idx = [X.columns.get_loc(c) for c in cat_cols]

X.head()


,High Blood Pressure,Told High Cholesterol,Cholesterol Checked,Body Mass Index,Smoked 100+ Cigarettes,Diagnosed Stroke,Diagnosed Diabetes,Leisure Physical Activity,Heavy Alcohol Consumption,Health Care Coverage,Doctor Visit Cost Barrier,General Health,Difficulty Walking,Sex,Education Level,Income Level,Age,Vegetable or Fruit Intake (1+ per Day)
0,Yes,Yes,Yes,40.68,Yes,No,No,No,No,Yes,No,Very Poor,Yes,Female,High school graduate,"$15,000 to less than $20,000",64,Yes
1,No,No,No,24.36,Yes,No,No,Yes,No,No,Yes,Fair,No,Female,College graduate,"Less than $10,000",50,No
2,Yes,Yes,Yes,27.33,No,No,No,No,No,Yes,Yes,Very Poor,Yes,Female,High school graduate,"$75,000 or more",61,Yes
3,Yes,No,Yes,27.01,No,No,No,Yes,No,Yes,No,Good,No,Female,Some high school,"$35,000 to less than $50,000",74,Yes
5,Yes,Yes,Yes,25.11,Yes,No,No,Yes,No,Yes,No,Good,No,Male,College graduate,"$75,000 or more",67,Yes


Cell 12

In [ ]:
params = {
    "loss_function": "Logloss",
    "eval_metric": "Logloss",
    "iterations": 2500,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 8,
    "random_strength": 1.0,
    "bagging_temperature": 1.0,
    "border_count": 128,
    "auto_class_weights": "Balanced",
    "grow_policy": "SymmetricTree",
    "random_seed": 42,
    "verbose": 200
}
params


{'loss_function': 'Logloss',
 'eval_metric': 'Logloss',
 'iterations': 2500,
 'learning_rate': 0.05,
 'depth': 6,
 'l2_leaf_reg': 8,
 'random_strength': 1.0,
 'bagging_temperature': 1.0,
 'border_count': 128,
 'auto_class_weights': 'Balanced',
 'grow_policy': 'SymmetricTree',
 'random_seed': 42,
 'verbose': 200}

Cell 13

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []


Cell 14

In [ ]:
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_va, y_va, cat_features=cat_features_idx)
    test_pool = Pool(X_test, cat_features=cat_features_idx)

    model = CatBoostClassifier(**params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
        early_stopping_rounds=300
    )

    oof_pred[va_idx] = model.predict_proba(valid_pool)[:, 1]
    test_pred += model.predict_proba(test_pool)[:, 1] / skf.n_splits

    fold_ap = average_precision_score(y_va, oof_pred[va_idx])
    fold_scores.append(fold_ap)
    print(f"Fold {fold} PR-AUC: {fold_ap:.6f}")



========== Fold 1 ==========
0:	learn: 0.6708729	test: 0.6710610	best: 0.6710610 (0)	total: 343ms	remaining: 14m 16s
200:	learn: 0.4522583	test: 0.4584280	best: 0.4584278 (199)	total: 45s	remaining: 8m 34s
400:	learn: 0.4445201	test: 0.4571851	best: 0.4571851 (400)	total: 1m 37s	remaining: 8m 27s
600:	learn: 0.4388515	test: 0.4580091	best: 0.4571851 (400)	total: 2m 35s	remaining: 8m 10s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.4571850745
bestIteration = 400

Shrink model to first 401 iterations.
Fold 1 PR-AUC: 0.365130

========== Fold 2 ==========
0:	learn: 0.6711575	test: 0.6711064	best: 0.6711064 (0)	total: 564ms	remaining: 23m 29s
200:	learn: 0.4523987	test: 0.4580952	best: 0.4580952 (200)	total: 50.1s	remaining: 9m 32s
400:	learn: 0.4456276	test: 0.4577026	best: 0.4575551 (339)	total: 1m 48s	remaining: 9m 25s
600:	learn: 0.4408546	test: 0.4581255	best: 0.4575551 (339)	total: 2m 50s	remaining: 8m 57s
Stopped by overfitting detector  (300 iterations wait

Cell 15

In [ ]:
print("Mean fold PR-AUC:", np.mean(fold_scores))
print("Std fold PR-AUC:", np.std(fold_scores))


Mean fold PR-AUC: 0.37025760879440905
Std fold PR-AUC: 0.003409572501083682


Cell 16

In [ ]:
thresholds = np.linspace(0.01, 0.60, 300)

rows = []
for t in thresholds:
    pred = (oof_pred >= t).astype(int)
    f2 = fbeta_score(y, pred, beta=2)
    p = precision_score(y, pred, zero_division=0)
    r = recall_score(y, pred, zero_division=0)
    rows.append((t, f2, p, r))

th_df = pd.DataFrame(rows, columns=["threshold", "f2", "precision", "recall"])
th_df = th_df.sort_values("f2", ascending=False).reset_index(drop=True)

best_threshold = th_df.loc[0, "threshold"]
best_f2 = th_df.loc[0, "f2"]

print("Best threshold:", best_threshold)
print("Best OOF F2:", best_f2)
display(th_df.head(10))


Best threshold: 0.5743478260869564
Best OOF F2: 0.54467251176628


,threshold,f2,precision,recall
0,0.574348,0.544673,0.252638,0.766050
1,0.572375,0.544568,0.251783,0.767766
2,0.576321,0.544280,0.253206,0.763781
3,0.570401,0.544234,0.250812,0.769205
4,0.578294,0.544224,0.253928,0.762010
5,0.568428,0.544196,0.250022,0.770976
6,0.580268,0.544161,0.254601,0.760350
7,0.566455,0.544054,0.249201,0.772581
8,0.582241,0.543984,0.255330,0.758302
9,0.584214,0.543948,0.255996,0.756752


Cell 17

In [ ]:
final_model = CatBoostClassifier(**params)

final_train_pool = Pool(X, y, cat_features=cat_features_idx)
final_test_pool = Pool(X_test, cat_features=cat_features_idx)

final_model.fit(final_train_pool, verbose=200)


0:	learn: 0.6702896	total: 425ms	remaining: 17m 42s
200:	learn: 0.4529788	total: 1m 1s	remaining: 11m 46s
400:	learn: 0.4465505	total: 2m 9s	remaining: 11m 20s
600:	learn: 0.4420840	total: 3m 21s	remaining: 10m 37s
800:	learn: 0.4376305	total: 4m 34s	remaining: 9m 42s
1000:	learn: 0.4336155	total: 5m 46s	remaining: 8m 39s
1200:	learn: 0.4295106	total: 7m	remaining: 7m 34s
1400:	learn: 0.4262349	total: 8m 15s	remaining: 6m 28s
1600:	learn: 0.4232360	total: 9m 31s	remaining: 5m 20s
1800:	learn: 0.4200325	total: 10m 46s	remaining: 4m 10s
2000:	learn: 0.4170166	total: 12m 2s	remaining: 3m
2200:	learn: 0.4140057	total: 13m 18s	remaining: 1m 48s
2400:	learn: 0.4110010	total: 14m 34s	remaining: 36.1s
2499:	learn: 0.4097387	total: 15m 11s	remaining: 0us


CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=1.0, border_count=128, depth=6, eval_metric='Logloss', grow_policy='SymmetricTree', iterations=2500, l2_leaf_reg=8, learning_rate=0.05, loss_function='Logloss', random_seed=42, random_strength=1.0, verbose=200)

Cell 18

In [ ]:
final_test_prob = final_model.predict_proba(final_test_pool)[:, 1]
final_test_label = np.where(final_test_prob >= best_threshold, "Yes", "No")

print(pd.Series(final_test_label).value_counts())

No     53577
Yes    20784
Name: count, dtype: int64


Cell 19

In [ ]:
submission = sample_sub.copy()
submission.iloc[:, 0] = test.iloc[:, 0].values
submission.iloc[:, 1] = final_test_label

submission_path = "/content/submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print("saved:", submission_path)


,ID,History of HeartDisease or Attack
0,test_000001,No
1,test_000002,No
2,test_000003,Yes
3,test_000004,No
4,test_000005,No


saved: /content/submission.csv
